# 04 · Baseline


Copy this notebook to start a model notebook. Change the two values in the first code cell, run the
second, and you hold the same frames as everyone else who names the same city and snapshot
([`docs/pipeline.md`](../docs/pipeline.md)). Everything below the second cell is yours to change.

Before the first run, download the snapshot (from the repo root):

```bash
python scripts/download_data.py toronto@2026-06-15
```


In [123]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="Pandas requires version")  # Anaconda 'bottleneck' noise
# `pip install -e .` once makes the next line unnecessary; it only finds the repo root when the package is not installed
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "strfair").is_dir())))
import strfair

CITY = "toronto"           # one of strfair.PINNED_SNAPSHOTS
SNAPSHOT = "2026-09-16"    # the team's date for that city (required: there is no "latest")

In [125]:
data = strfair.prepare(CITY, SNAPSHOT)   # compare the `data` hash with a teammate: same hash, same data

toronto 2026-09-16 | listings 22,050 rows, 16,760 in sample | features 106 cols | code 025a7a1b | data 93c4b714  (cached)


## The frames

One DataFrame per Inside Airbnb file. `listings` (with the `in_sample` flag) is the contract; `features`
is a starting point built from the in-sample rows, in the same row order as `data.sample`. The other
files load on first access.

In [126]:
print(f"listings {data.listings.shape}   sample {data.sample.shape}   features {data.features.shape}")
data.features.head(3)

listings (22050, 93)   sample (16760, 93)   features (16760, 106)


,id,city,snapshot,price,log_price,lead_days,quote_nights,weekend_checkin,checkin_dow,checkin_month,...,amg_family_pet,amg_exercise,amg_parking,amg_bathroom_laundry,amg_connectivity,am_wifi,am_free_parking,am_smoking_allowed,am_breakfast,price_per_person
0,1092352202113212073,toronto,2026-09-16,54.26,3.993787,24,30,0.0,0,10,...,0,0,0,1,1,1,0,0,0,27.130
1,1066654035322408628,toronto,2026-09-16,41.86,3.734331,1,30,1.0,4,9,...,0,0,0,0,1,1,0,0,0,41.860
2,771633308407367926,toronto,2026-09-16,67.89,4.217889,0,28,1.0,4,9,...,0,0,0,1,1,1,0,0,0,33.945


In [127]:
# which builder each standard feature comes from (strfair/features.py); add your own columns freely
strfair.feature_families(data.sample).value_counts()

family
reviews         20
target_quote    16
amenities       14
host            12
listing_text    12
availability    12
size            10
location         6
Name: count, dtype: int64

## From here on it is your notebook

Pick columns, split or cross-validate, impute, encode, fit. A few helpers you may want:

- `strfair.LEAKY`: columns that are functions of the price itself (targets, revenue, value score). Never inputs.
- `strfair.DEMAND_RESPONSE`: availability and recent-review columns. The proposal uses them to validate labels, not as inputs.
- `strfair.CATEGORICAL`: the categorical columns among the standard features.
- `strfair.impute_sizes(df, train=mask)`: median-by-room-type fill for `bedrooms` / `beds` / `baths` plus `_missing` flags.

Because every notebook holds the identical frame, `KFold(5, shuffle=True, random_state=42)` gives
everyone the same folds without any shared split column. Out-of-fold predictions give a residual for
every listing, which is what the fair-price labels need.

## 1. Setup

In [128]:
SNAPSHOTS = {"new-york-city": "2026-09-14", 
                "los-angeles": "2026-09-14", 
                "toronto": "2026-09-16", 
                "san-diego": "2026-09-20",
                "clark-county-nv": "2026-09-20", 
                "jersey-city": "2026-09-21"}

SEED, N_SPLITS = 42, 5 # 

city = "jersey-city"
data = strfair.prepare(city, SNAPSHOTS[city])

jersey-city 2026-09-21 | listings 2,330 rows, 2,093 in sample | features 106 cols | code 025a7a1b | data 460245b9  (cached)


## 2. Create X, y, groups

In [129]:
# drop discount_amount because it's also leakage
# discount_amount is the quote's discount in dollars (≈ price × quote_nights × discount_share / (1 − discount_share))

F = data.features
drop = strfair.LEAKY + strfair.DEMAND_RESPONSE + strfair.ID_COLUMNS + ["neighbourhood_raw", "discount_amount"]  # raw = unpooled copy of `neighbourhood`
X_cols = [c for c in F.columns if c not in drop]

X, y = F[X_cols], F["log_price"]

display(X_cols)
print(len(X_cols))

['lead_days',
 'quote_nights',
 'weekend_checkin',
 'checkin_dow',
 'checkin_month',
 'is_monthly_quote',
 'lead_band',
 'has_discount',
 'discount_share',
 'minimum_nights',
 'min_nights_band',
 'room_type',
 'room',
 'property_group',
 'accommodates',
 'bedrooms',
 'beds',
 'shared_bath',
 'baths',
 'guests_per_bedroom',
 'guests_per_bed',
 'latitude',
 'longitude',
 'km_to_centre',
 'log_km_to_centre',
 'neighbourhood',
 'superhost',
 'host_identity_verified',
 'host_has_profile_pic',
 'host_listings_count',
 'calculated_host_listings_count',
 'calculated_host_listings_count_entire_homes',
 'calculated_host_listings_count_private_rooms',
 'log_host_listings',
 'multi_host',
 'single_listing_host',
 'host_entire_home_share',
 'host_years',
 'number_of_reviews',
 'log_reviews',
 'no_reviews',
 'tenure_months',
 'new_listing',
 'months_since_last_review',
 'review_scores_rating',
 'review_scores_accuracy',
 'review_scores_cleanliness',
 'review_scores_checkin',
 'review_scores_communic

79


## 3. Baseline 0 Jersey City

Predict the median log price of the training set for every listing. No features are used, so any
real model should beat this.

**Why use median?**

- If we predict one number for every listing, the median gives the lowest MAE.
- A few very expensive listings pull the mean up, but not the median.
- The median of log price is the log of the median price, so the guess is just the typical nightly
  price (about $216 in Jersey City).

In [130]:
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

# host of each listing, so one host's listings stay in the same fold
groups = data.sample["host_id"].values
gkf = GroupKFold(n_splits=5) # GroupKFold

maes = []                     # MAE of each fold
oof_b0 = np.zeros(len(y))     # prediction for every listing, filled fold by fold

for train_idx, test_idx in gkf.split(X, y, groups):
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    # predict the training median for every test listing
    pred = np.full(len(y_test), y_train.median())

    oof_b0[test_idx] = pred
    maes.append(mean_absolute_error(y_test, pred))

print("MAE per fold:", np.round(maes, 3))
print(f"Baseline 0 MAE: {np.mean(maes):.3f} (+/- {np.std(maes):.3f})")

# same error back in dollars
print(f"MAE in dollars: ${mean_absolute_error(np.exp(y), np.exp(oof_b0)):.0f}")

MAE per fold: [0.621 0.731 0.69  0.668 0.689]
Baseline 0 MAE: 0.680 (+/- 0.036)
MAE in dollars: $157


**Result** (Jersey City, 2026-09-21)

- MAE on log price: 0.680 ± 0.036 over 5 folds (0.62 to 0.73).
- A log error of 0.68 means the guess is off by about 2x (e^0.68 ≈ 2), so roughly half or double the real price.
- MAE in dollars: $157 a night.
- Any model that uses listing features has to beat this.

## 4. Baseline 1 for Jersey City

Median across listings in the same neighbourhood and room type: for each listing, predict the median
log price of the training listings that share its neighbourhood and room type.

**Why neighbourhood and room type?**

- This is roughly how hosts price: look at similar places nearby.
- Room type and neighbourhood are two of the strongest single columns in `docs/features.md`.
- It uses only two columns, so the gap to the real models shows what the other features add.

**Small groups:** a group with fewer than 5 training listings is too noisy, so those listings get the
neighbourhood median, then the room type median, then the city median.

In [131]:
import pandas as pd

maes = []                     # MAE of each fold
oof_b1 = np.zeros(len(y))     # prediction for every listing, filled fold by fold
n_fallback = 0                # test listings whose group was too small in the training set

# same GroupKFold as Baseline 0, so both are scored on the same folds
for train_idx, test_idx in gkf.split(X, y, groups):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # group label for each listing, e.g. "Downtown | Entire home/apt"
    group_train = X_train["neighbourhood"] + " | " + X_train["room_type"]
    group_test = X_test["neighbourhood"] + " | " + X_test["room_type"]

    # median log price of each group in the training set, only for groups with 5+ listings
    counts = group_train.value_counts()
    big = group_train.isin(counts[counts >= 5].index)
    group_median = y_train[big].groupby(group_train[big]).median()

    pred = group_test.map(group_median)
    n_fallback += pred.isna().sum()

    # if the group is missing: neighbourhood median, then room type median, then city median
    pred = pred.fillna(X_test["neighbourhood"].map(y_train.groupby(X_train["neighbourhood"]).median()))
    pred = pred.fillna(X_test["room_type"].map(y_train.groupby(X_train["room_type"]).median()))
    pred = pred.fillna(y_train.median())

    oof_b1[test_idx] = pred
    maes.append(mean_absolute_error(y_test, pred))

print("MAE per fold:", np.round(maes, 3))
print(f"Baseline 1 MAE: {np.mean(maes):.3f} (+/- {np.std(maes):.3f})")
print(f"MAE in dollars: ${mean_absolute_error(np.exp(y), np.exp(oof_b1)):.0f}")
print(f"Listings that used a fallback: {n_fallback} of {len(y)}")

MAE per fold: [0.435 0.509 0.463 0.436 0.489]
Baseline 1 MAE: 0.466 (+/- 0.029)
MAE in dollars: $117
Listings that used a fallback: 2 of 2093


**Result** (Jersey City, 2026-09-21)

- MAE on log price: 0.466 ± 0.029 over 5 folds (0.44 to 0.51).
- e^0.466 ≈ 1.6, so the guess is typically off by about 60%.
- MAE in dollars: $117 a night (Baseline 0: $157).
- 31% lower MAE than Baseline 0, using only two columns.
- Only 2 listings needed a fallback (the city's 2 shared rooms).

## 5. Baseline 0 & Baseline 1 (All six cities)

In [132]:
import warnings
warnings.filterwarnings("ignore", message=".*not a team snapshot.*")

drop = strfair.LEAKY + strfair.DEMAND_RESPONSE + strfair.ID_COLUMNS + ["neighbourhood_raw", "discount_amount"]
gkf = GroupKFold(n_splits=5)

def baseline0(X, y, groups):
    maes = []
    for train_idx, test_idx in gkf.split(X, y, groups):
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]
        pred = np.full(len(y_test), y_train.median())
        maes.append(mean_absolute_error(y_test, pred))
    return np.mean(maes)


def baseline1(X, y, groups):
    maes = []
    for train_idx, test_idx in gkf.split(X, y, groups):
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        group_train = X_train["neighbourhood"] + " | " + X_train["room_type"]
        group_test = X_test["neighbourhood"] + " | " + X_test["room_type"]

        # median of each group with at least 5 listings
        counts = group_train.value_counts()
        big = group_train.isin(counts[counts >= 5].index)
        group_median = y_train[big].groupby(group_train[big]).median()

        # small or unseen groups: neighbourhood, then room type, then city median
        pred = group_test.map(group_median)
        pred = pred.fillna(X_test["neighbourhood"].map(y_train.groupby(X_train["neighbourhood"]).median()))
        pred = pred.fillna(X_test["room_type"].map(y_train.groupby(X_train["room_type"]).median()))
        pred = pred.fillna(y_train.median())

        maes.append(mean_absolute_error(y_test, pred))
    return np.mean(maes)


rows = []
for city, snapshot in SNAPSHOTS.items():
    data = strfair.prepare(city, snapshot)
    F = data.features
    X = F[[c for c in F.columns if c not in drop]]
    y = F["log_price"]
    groups = data.sample["host_id"].values

    b0 = baseline0(X, y, groups)
    b1 = baseline1(X, y, groups)
    rows.append([city, len(y), b0, b1])

results = pd.DataFrame(rows, columns=["city", "listings", "baseline0_mae", "baseline1_mae"])
results["improvement"] = 1 - results["baseline1_mae"] / results["baseline0_mae"]
results.round(3)

new-york-city 2026-09-14 | listings 29,918 rows, 19,396 in sample | features 107 cols | code 025a7a1b | data 8d904721  (cached)
los-angeles 2026-09-14 | listings 43,781 rows, 36,663 in sample | features 107 cols | code 025a7a1b | data c42b65f1  (cached)
toronto 2026-09-16 | listings 22,050 rows, 16,760 in sample | features 106 cols | code 025a7a1b | data 93c4b714  (cached)
san-diego 2026-09-20 | listings 13,253 rows, 11,614 in sample | features 106 cols | code 025a7a1b | data 23c71b6d  (cached)
clark-county-nv 2026-09-20 | listings 20,651 rows, 17,440 in sample | features 106 cols | code 025a7a1b | data 2a715a57  (cached)
jersey-city 2026-09-21 | listings 2,330 rows, 2,093 in sample | features 106 cols | code 025a7a1b | data 460245b9  (cached)


,city,listings,baseline0_mae,baseline1_mae,improvement
0,new-york-city,19396,0.687,0.528,0.231
1,los-angeles,36663,0.710,0.524,0.262
2,toronto,16760,0.686,0.565,0.177
3,san-diego,11614,0.611,0.514,0.159
4,clark-county-nv,17440,0.641,0.592,0.077
5,jersey-city,2093,0.680,0.466,0.314


**Result** (2026-09 snapshots, GroupKFold by host)

- Baseline 1 beats Baseline 0 in every city.
- Clark County gains the least (8%): like Jersey City, its neighbourhoods are a few large areas with little
  price information. These two cities will need better location features.
- LA and NYC have many small neighbourhoods, so the neighbourhood + room type median helps more there.

## 6. Model 1: Elastic net

Linear model on all candidate features in `X` (79 columns, 80 in NYC and LA), with the settings from
`notebooks/00`. No tuning yet: this is the reference that feature selection is compared against.

**Why Elastic net?**

- It is the interpretable model in `docs/models.md`: a coefficient is roughly a % change in price.
- The L1 + L2 penalty copes with many correlated features and can shrink weak ones to zero.

**Preprocessing** (refit on the training rows of every fold): numbers get the median for missing values
and are scaled; categories become one column per level, with levels under 20 listings grouped.

In [133]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def make_enet(X):
    cat = list(X.select_dtypes(exclude="number").columns)
    num = [c for c in X.columns if c not in cat]

    # numbers: fill missing values with the median, then scale
    # categories: one column per level, levels with < 20 listings are grouped together
    pre = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), cat),
    ])
    return make_pipeline(pre, ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000))

## 7. Model 2: Random forest

Many decision trees on the same features, averaged. Also untuned: 300 trees, at least 5 listings per leaf.

**Why random forest?**

- It is the ensemble model in `docs/models.md`.
- It picks up non-linear effects and interactions (for example latitude with longitude) that a linear
  model misses, so the gap to Elastic net shows how much accuracy the interpretable model gives up.

**Preprocessing:** trees need no scaling or one-hot columns, so each category is just numbered.

In [134]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OrdinalEncoder


def make_rf(X):
    cat = list(X.select_dtypes(exclude="number").columns)
    num = [c for c in X.columns if c not in cat]

    # trees don't need scaling or one-hot columns: each category just gets a number
    pre = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num),
        ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                              OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)), cat),
    ])
    return make_pipeline(pre, RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                                    n_jobs=-1, random_state=42))

## 8. All four models (six cities)

In [135]:
# baseline0, baseline1, drop and gkf come from section 5; make_enet and make_rf from sections 6 and 7
def run_model(make_model, X, y, groups):
    maes = []
    for train_idx, test_idx in gkf.split(X, y, groups):
        model = make_model(X)    # a new model each fold
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        pred = model.predict(X.iloc[test_idx])
        maes.append(mean_absolute_error(y.iloc[test_idx], pred))
    return np.mean(maes)


rows = []
for city, snapshot in SNAPSHOTS.items():
    data = strfair.prepare(city, snapshot)
    F = data.features
    X = F[[c for c in F.columns if c not in drop]]
    y = F["log_price"]
    groups = data.sample["host_id"].values

    b0 = baseline0(X, y, groups)
    b1 = baseline1(X, y, groups)
    m1 = run_model(make_enet, X, y, groups)
    m2 = run_model(make_rf, X, y, groups)
    rows.append([city, len(y), b0, b1, m1, m2])

results = pd.DataFrame(rows, columns=["city", "listings", "baseline0_mae", "baseline1_mae",
                                      "elastic_net_mae", "random_forest_mae"])
results.round(3)

new-york-city 2026-09-14 | listings 29,918 rows, 19,396 in sample | features 107 cols | code 025a7a1b | data 8d904721  (cached)
los-angeles 2026-09-14 | listings 43,781 rows, 36,663 in sample | features 107 cols | code 025a7a1b | data c42b65f1  (cached)
toronto 2026-09-16 | listings 22,050 rows, 16,760 in sample | features 106 cols | code 025a7a1b | data 93c4b714  (cached)
san-diego 2026-09-20 | listings 13,253 rows, 11,614 in sample | features 106 cols | code 025a7a1b | data 23c71b6d  (cached)
clark-county-nv 2026-09-20 | listings 20,651 rows, 17,440 in sample | features 106 cols | code 025a7a1b | data 2a715a57  (cached)
jersey-city 2026-09-21 | listings 2,330 rows, 2,093 in sample | features 106 cols | code 025a7a1b | data 460245b9  (cached)


,city,listings,baseline0_mae,baseline1_mae,elastic_net_mae,random_forest_mae
0,new-york-city,19396,0.687,0.528,0.295,0.280
1,los-angeles,36663,0.710,0.524,0.284,0.267
2,toronto,16760,0.686,0.565,0.252,0.245
3,san-diego,11614,0.611,0.514,0.283,0.276
4,clark-county-nv,17440,0.641,0.592,0.342,0.305
5,jersey-city,2093,0.680,0.466,0.226,0.229


**Result** (2026-09 snapshots, GroupKFold by host, MAE on log price)

- Both models cut the error roughly in half compared with Baseline 1, in every city.
- Random forest is a little better than Elastic net in 5 of 6 cities. The biggest gap is Clark County
  (0.342 vs 0.305), where the forest can use latitude and longitude directly. Elsewhere the gap is at most
  0.017, so the interpretable model gives up little accuracy.